# MedXIR retrieval: Qwen3-Embedding-4B, exact search over the stored vectors

Generated by `scripts/build_retrieval_notebook.py`. Edit the generator, not this file.

Run on RTX Pro 6000 with Internet OFF. Attach the three vector datasets, the retrieval inputs,
the chunk corpus kernel output and the package model kernel output.

In [ ]:
import faulthandler
import json
import os
import resource
import time
import traceback
from contextlib import contextmanager
from pathlib import Path

OUT = Path('/kaggle/working')
OUT.mkdir(parents=True, exist_ok=True)
T0 = time.perf_counter()
PROGRESS = OUT / 'progress.txt'
FAULT_FILE = open(OUT / 'faulthandler.txt', 'w')  # native crashes dump the Python stack here
faulthandler.enable(file=FAULT_FILE, all_threads=True)
FAILED = None


def note(message):
    rss_gib = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2 ** 20  # Linux reports KiB
    line = f'{time.perf_counter() - T0:8.1f}s  peak_rss={rss_gib:.1f}GiB  {message}'
    print(line, flush=True)
    with open(PROGRESS, 'a', encoding='utf-8') as file:
        file.write(line + '\n')


@contextmanager
def stage(name):
    # A Python exception is written to retrieval_error.txt and does not stop the kernel, so the
    # session ends COMPLETE and the files can be fetched; later steps are skipped.
    global FAILED
    note(f'start {name}')
    try:
        yield
        note(f'done {name}')
    except Exception:
        FAILED = name
        (OUT / 'retrieval_error.txt').write_text(f'stage: {name}\n\n{traceback.format_exc()}', encoding='utf-8')
        note(f'FAILED {name}; traceback in retrieval_error.txt')

## 1. Inputs and GPU check

In [ ]:
if FAILED is None:
    with stage('setup'):
        import glob
        import subprocess
        import zipfile

        import numpy as np
        import pandas as pd
        import pyarrow.parquet as pq
        import torch

        MODEL_DIR = 'Qwen3-Embedding-4B'
        TASK = 'Given a web search query, retrieve relevant passages that answer the query'
        MAX_LENGTH = 1024
        EMBED_BATCH = 8
        SEARCH_BATCH = 64
        TOP_CHUNKS = 20  # chunks per query; not tuned yet


        def one_match(pattern, description):
            hits = sorted(glob.glob(pattern, recursive=True))
            assert len(hits) == 1, f'{description}: expected one match, found {hits[:10]}'
            return Path(hits[0])


        QUERY_PATH = one_match('/kaggle/input/**/query.parquet', 'query.parquet')
        CORPUS_PATH = one_match('/kaggle/input/**/links_corpus_vi_en_zh.parquet', 'official corpus IDs')
        HASH_PATH = one_match('/kaggle/input/**/hash_groups.parquet', 'hash_groups.parquet')
        MODEL_PATH = one_match(f'/kaggle/input/**/{MODEL_DIR}/config.json', 'packaged Qwen model').parent
        VECTOR_FILES = sorted(glob.glob('/kaggle/input/**/vectors/*.npy', recursive=True))
        assert len(VECTOR_FILES) == 9, f'expected 9 vector parts, found {len(VECTOR_FILES)}'
        VECTOR_PATH = {Path(path).stem: path for path in VECTOR_FILES}
        assert len(VECTOR_PATH) == 9, 'duplicate vector part names across the attached datasets'
        CHUNK_FILES = sorted(glob.glob('/kaggle/input/**/chunks/*/part-*.parquet', recursive=True))
        assert len(CHUNK_FILES) == 9, f'expected 9 chunk parts, found {len(CHUNK_FILES)}'
        print('query:', QUERY_PATH, '| model:', MODEL_PATH)
        print(len(VECTOR_FILES), 'vector parts;', len(CHUNK_FILES), 'chunk parts')

        print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version', '--format=csv'],
                             capture_output=True, text=True).stdout)
        gpu_name = torch.cuda.get_device_name(0)
        assert '6000' in gpu_name, f'Expected RTX Pro 6000, got {gpu_name!r}'
        print('GPU:', gpu_name)

## 2. Encode the queries with the corpus instruction and pooling

In [ ]:
if FAILED is None:
    with stage('encode'):
        from transformers import AutoModel, AutoTokenizer
        import torch.nn.functional as F

        os.environ['HF_HUB_OFFLINE'] = '1'
        tok = AutoTokenizer.from_pretrained(str(MODEL_PATH))
        tok.padding_side = 'left'
        dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
        model = AutoModel.from_pretrained(str(MODEL_PATH), torch_dtype=dtype,
                                          attn_implementation='sdpa').cuda().eval()

        queries = pd.read_parquet(QUERY_PATH, columns=['id', 'query'])
        assert len(queries) == 1200 and queries['id'].is_unique
        texts = [f'Instruct: {TASK}\nQuery:{q}' for q in queries['query'].astype(str)]


        def last_token_pool(hidden, mask):
            if mask[:, -1].sum() == mask.shape[0]:
                return hidden[:, -1]
            lengths = mask.sum(dim=1) - 1
            return hidden[torch.arange(hidden.shape[0], device=hidden.device), lengths]


        @torch.inference_mode()
        def encode(texts):
            tokenized = tok(texts, truncation=True, max_length=MAX_LENGTH)['input_ids']
            order = np.argsort([-len(row) for row in tokenized], kind='stable')
            vectors = np.empty((len(texts), model.config.hidden_size), dtype=np.float32)
            for start in range(0, len(order), EMBED_BATCH):
                index = order[start:start + EMBED_BATCH]
                batch = tok.pad({'input_ids': [tokenized[i] for i in index]}, return_tensors='pt').to('cuda')
                hidden = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'],
                               use_cache=False).last_hidden_state
                pooled = F.normalize(last_token_pool(hidden, batch['attention_mask']).float(), p=2, dim=1)
                vectors[index] = pooled.cpu().numpy()
            return vectors


        query_vectors = encode(texts)
        assert query_vectors.shape == (1200, model.config.hidden_size)
        assert np.isfinite(query_vectors).all()
        del model, tok
        torch.cuda.empty_cache()
        print('encoded', len(queries), 'queries | elapsed', round(time.perf_counter() - T0, 1), 's')

## 3. Exact search, one vector part at a time

In [ ]:
if FAILED is None:
    with stage('search'):
        torch.backends.cuda.matmul.allow_tf32 = False
        query_gpu = torch.from_numpy(query_vectors).cuda()
        per_query = [[] for _ in range(len(queries))]  # (score, part key, row)
        started = time.perf_counter()
        for key, path in VECTOR_PATH.items():
            vectors = torch.from_numpy(np.load(path)).cuda().float()
            assert vectors.shape[1] == query_vectors.shape[1] and torch.isfinite(vectors).all(), key
            k = min(TOP_CHUNKS, vectors.shape[0])
            for start in range(0, len(queries), SEARCH_BATCH):
                scores, rows = torch.topk(query_gpu[start:start + SEARCH_BATCH] @ vectors.T, k=k, dim=1)
                scores, rows = scores.cpu().numpy(), rows.cpu().numpy()
                for offset in range(len(scores)):
                    per_query[start + offset].extend(
                        (float(score), key, int(row)) for score, row in zip(scores[offset], rows[offset]))
            print(key, 'rows', f'{vectors.shape[0]:,}', '| elapsed', round(time.perf_counter() - started, 1), 's')
            del vectors
            torch.cuda.empty_cache()
        del query_gpu
        for qi in range(len(per_query)):
            per_query[qi].sort(key=lambda item: item[0], reverse=True)
            per_query[qi] = per_query[qi][:TOP_CHUNKS]
        print('search done in', round(time.perf_counter() - started, 1), 's')

## 4. Join candidates to chunk text and document groups

In [ ]:
if FAILED is None:
    with stage('join'):
        # Map (part, row) to chunk_id through the ids file, then read only the candidate rows.
        wanted_rows = {}
        for query_rows in per_query:
            for _, key, row in query_rows:
                wanted_rows.setdefault(key, set()).add(row)

        chunk_id_at = {}
        for key, rows in wanted_rows.items():
            # The ids file sits next to its .npy, in the same dataset (the three shards are separate datasets).
            ids_path = Path(VECTOR_PATH[key]).with_name(f'{key}-ids.parquet')
            column = pq.read_table(ids_path, columns=['chunk_id']).column('chunk_id').to_pylist()
            chunk_id_at[key] = {row: column[row] for row in rows}

        wanted_ids = {cid for mapping in chunk_id_at.values() for cid in mapping.values()}
        records = {}
        for path in CHUNK_FILES:
            table = pq.read_table(path, columns=['chunk_id', 'text_sha256', 'rep_doc_id', 'lang', 'order', 'chunk_text'])
            names = table.column('chunk_id').to_pylist()
            keep = [i for i, cid in enumerate(names) if cid in wanted_ids]
            for i in keep:
                records[names[i]] = dict(
                    text_sha256=table.column('text_sha256')[i].as_py(),
                    rep_doc_id=int(table.column('rep_doc_id')[i].as_py()),
                    chunk_order=int(table.column('order')[i].as_py()),
                    chunk_text=table.column('chunk_text')[i].as_py(),
                )
            del table
        missing = wanted_ids - set(records)
        assert not missing, f'{len(missing)} candidate chunk_ids have no source text, e.g. {sorted(missing)[:3]}'

        groups = pd.read_parquet(HASH_PATH, columns=['text_sha256', 'doc_ids', 'fanout_safe'])
        # doc_ids comes back as a numpy array: convert to a plain list so truthiness and iteration are safe.
        group_lookup = {
            sha: ([int(doc) for doc in doc_ids] if bool(safe) else None)
            for sha, doc_ids, safe in zip(groups['text_sha256'], groups['doc_ids'], groups['fanout_safe'])
        }
        print('candidate chunks:', len(records), '| hash groups:', len(group_lookup))

## 5. Write and validate the submission

In [ ]:
if FAILED is None:
    with stage('submit'):
        submission = []
        empty = 0
        for qi, query_id in enumerate(queries['id'].tolist()):
            chunks, relevant_docs, seen = [], [], set()
            for _, key, row in per_query[qi]:
                cid = chunk_id_at[key][row]
                record = records[cid]
                doc_ids = group_lookup.get(record['text_sha256'])
                if not doc_ids:
                    doc_ids = [record['rep_doc_id']]
                for doc_id in doc_ids:
                    if doc_id not in seen:
                        seen.add(doc_id)
                        relevant_docs.append(doc_id)
                    chunks.append(dict(doc_id=doc_id, chunk_text=record['chunk_text'],
                                       chunk_order=record['chunk_order']))
            if not chunks:
                empty += 1
            submission.append(dict(id=int(query_id), relevant_docs=relevant_docs, relevant_chunks=chunks))

        corpus_ids = set(map(int, pd.read_parquet(CORPUS_PATH, columns=['id'])['id'].tolist()))
        assert len(submission) == 1200 and {row['id'] for row in submission} == set(map(int, queries['id'].tolist()))
        for entry in submission:
            assert set(entry['relevant_docs']).issubset(corpus_ids), entry['id']
            listed = set(entry['relevant_docs'])
            assert all(chunk['doc_id'] in listed and chunk['chunk_text'].strip() for chunk in entry['relevant_chunks'])

        json_path = OUT / 'medxir-qwen3-4b-submission.json'
        zip_path = OUT / 'medxir-qwen3-4b-submission.zip'
        json_path.write_text(json.dumps(submission, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
        with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
            archive.write(json_path, arcname=json_path.name)
        with zipfile.ZipFile(zip_path) as archive:
            assert archive.namelist() == [json_path.name]

        result = dict(
            model='Qwen/Qwen3-Embedding-4B', search='exact float32 matmul over stored float16 vectors',
            observed_hardware=gpu_name, query_count=len(submission), top_chunks=TOP_CHUNKS,
            empty_queries=empty, output_json=str(json_path), output_zip=str(zip_path),
            json_bytes=json_path.stat().st_size, zip_bytes=zip_path.stat().st_size,
            elapsed_seconds=round(time.perf_counter() - T0, 1),
        )
        (OUT / 'retrieval_manifest.json').write_text(json.dumps(result, indent=2) + '\n', encoding='utf-8')
        print(json.dumps(result, indent=2))

## 6. Report

In [ ]:
note('finished; status ' + ('error' if FAILED else 'ok'))
if FAILED:
    (OUT / 'retrieval_manifest.json').write_text(json.dumps(
        dict(status='error', failed_stage=FAILED, error_file='retrieval_error.txt'), indent=2) + '\n')